# Sprint 5 Report – Resampling and Ensemble Techniques
**Course:** Data Intensive Systems (4DV652)  
**Lab:** Lab Lecture 5  
**Deadline:** 2026-02-25  

---

## Overview

This sprint focused on applying **heterogeneous ensemble methods** and **stacking** to challenge the current regression and classification champions established in Sprint 4. The team worked across two parallel tracks:

- **Regression Ensemble** – Predict `AimoScore` (continuous target) using diverse base models with bootstrap sampling and aggregation strategies.
- **Classification Ensemble** – Predict `WeakestLink` (14-class target) using voting classifiers, bagging, and stacking.

A custom `CorrelationFilter` transformer was also implemented as a reusable sklearn-compatible preprocessing component shared across both tracks.

---

## 1. ML Process Iteration

### 1.1 Problem Framing Recap

The dataset originates from movement quality assessments (NASM Overhead Squat Assessment). Two supervised learning tasks are defined:

| Task | Target | Type | Sprint 4 Champion Score |
|------|--------|------|-------------------------|
| Regression | `AimoScore` (continuous) | Regression | R² = 0.6356, RMSE = 0.1303 |
| Classification | `WeakestLink` (14 classes) | Multiclass | Weighted F1 = 0.6110 |

### 1.2 Sprint 5 Goals

Per the lab assignment:
1. Define ensembles of independent models using bootstrap samples, different feature engineering, and diverse AI approaches.
2. Challenge the Sprint 4 champions using simple aggregation (averaging / majority vote) or stacking.
3. Deploy a new champion if the ensemble outperforms the previous one.
4. Validate improvement using the **corrected resampled t-test** (Nadeau & Bengio, 2003).

---

## 2. Software Development: CorrelationFilter

A custom `sklearn`-compatible transformer was implemented and used in both the regression and classification pipelines. It removes highly correlated features before model fitting, reducing redundancy while preserving sklearn `Pipeline` compatibility.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Removes features that are highly correlated with another feature.
    Threshold defaults to 0.99 (absolute Pearson correlation).
    Sklearn-compatible: can be used in Pipeline objects.
    """
    def __init__(self, threshold=0.99):
        self.threshold = threshold
        self.keep_cols_ = None

    def fit(self, X, y=None):
        Xdf = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X
        # Compute absolute correlation matrix (upper triangle only)
        corr = Xdf.corr(numeric_only=True).abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        to_drop = [col for col in upper.columns if any(upper[col] >= self.threshold)]
        self.keep_cols_ = [c for c in Xdf.columns if c not in to_drop]
        return self

    def transform(self, X):
        Xdf = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X
        return Xdf[self.keep_cols_].copy()

# Example usage
print("CorrelationFilter: removes columns with |corr| >= threshold.")
print("Used in regression pipeline with threshold=0.95 for RandomForest.")

---
## 3. Regression Ensemble (A5_Regression_Ensemble)

### 3.1 Approach

A heterogeneous ensemble was designed following the sprint lecture pattern:

- **Bootstrap diversity**: Four different bootstrap-augmented training sets (`dataset2_train_augmented_1..4.csv`).
- **Model diversity**: Four distinct regressors (Lasso, Ridge, RandomForest, GradientBoosting), each trained on a different bootstrap sample.
- **Feature diversity**: Feature subsets were defined (full, angle-only, NASM-only, angle+NASM) to allow further differentiation.
- **Aggregation**: Two strategies — simple averaging and CV-R²-weighted averaging.

### 3.2 Ensemble Configuration

| Model | Bootstrap Sample | Feature Set | Algorithm |
|-------|-----------------|-------------|----------|
| Lasso | 1 | Full | `LassoCV` (cv=5) |
| Ridge | 2 | Full | `RidgeCV` (cv=5) |
| RandomForest | 3 | Full | `RandomForestRegressor` (200 trees, depth=15) + CorrelationFilter(0.95) |
| GradientBoosting | 4 | Full | `GradientBoostingRegressor` (150 trees, depth=5, lr=0.1) |

In [ ]:
# Ensemble configuration (from A5_Regression_Ensemble.ipynb)
ENSEMBLE_CONFIG = [
    {"name": "lasso",  "bootstrap": 1, "features": "full"},
    {"name": "ridge",  "bootstrap": 2, "features": "full"},
    {"name": "rf",     "bootstrap": 3, "features": "full"},
    {"name": "gb",     "bootstrap": 4, "features": "full"},
]

# Feature subsets available (used for diversity)
FEATURE_SUBSETS = {
    "full":       "all features",
    "angle_only": "features with 'Angle' in name",
    "nasm_only":  "features with 'NASM' in name",
    "angle_nasm": "angle + NASM features (excludes time)",
}

print("Ensemble configuration loaded.")
print(f"Number of base models: {len(ENSEMBLE_CONFIG)}")

### 3.3 Base Model Results (Test Set)

Each base model was trained on its assigned bootstrap sample and evaluated on the held-out test set. Cross-validation R² scores were used to compute ensemble weights.

| Model | Bootstrap | CV R² | Test R² | Test RMSE | Test MAE |
|-------|-----------|-------|---------|-----------|----------|
| Lasso | 1 | — | — | — | — |
| Ridge | 2 | — | — | — | — |
| RandomForest | 3 | — | — | — | — |
| GradientBoosting | 4 | — | — | — | — |

> *Note: Exact numeric outputs are produced at runtime. The table above is populated when executing the notebook against the dataset.*

### 3.4 Ensemble Aggregation Results

| Method | R² | RMSE | MAE | Corr |
|--------|----|------|-----|------|
| Simple Average | runtime | runtime | runtime | runtime |
| Weighted Average (CV R²) | runtime | runtime | runtime | runtime |
| **A4 Champion (baseline)** | **0.6356** | **0.1303** | **0.0972** | **0.8089** |

Weighted averaging assigns higher influence to models with better cross-validation R² scores:
$$\hat{y}_{\text{ensemble}} = \sum_{i=1}^{4} w_i \hat{y}_i, \quad w_i = \frac{\text{CV-R}^2_i}{\sum_j \text{CV-R}^2_j}$$

### 3.5 Statistical Significance – Corrected Resampled t-test

Standard paired t-tests overstate confidence when models are compared via cross-validation because training folds overlap. The **Nadeau & Bengio (2003) correction** accounts for this by inflating the variance:

$$\text{Var}_{\text{corrected}} = \left(\frac{1}{k} + \frac{n_{\text{test}}}{n_{\text{train}}}\right) \cdot \hat{\sigma}^2_{\Delta}$$

For this dataset, $n_{\text{test}}/n_{\text{train}} \approx 0.25$, meaning variance is inflated by ~25%, making it harder to claim statistically significant improvement.

Hypotheses tested (α = 0.05):
- H₀: Ensemble MSE = Champion MSE  
- H₁: Ensemble MSE ≠ Champion MSE (two-tailed)

| Comparison | t-stat | p-value | Significant? |
|------------|--------|---------|-------------|
| Simple Avg vs Champion | runtime | runtime | runtime |
| Weighted Avg vs Champion | runtime | runtime | runtime |

In [ ]:
# Corrected resampled t-test implementation (Nadeau & Bengio, 2003)
import numpy as np
from scipy import stats

def corrected_resampled_ttest(errors_1, errors_2, n_train, n_test):
    """
    Corrected resampled t-test for comparing two models.
    Accounts for variance inflation due to overlapping cross-validation folds.
    
    Args:
        errors_1, errors_2: arrays of per-sample squared errors for model 1 and 2
        n_train: number of training samples
        n_test:  number of test samples
    Returns:
        t_stat, p_value, mean_diff
    """
    diff = errors_1 - errors_2
    mean_diff = np.mean(diff)
    var_diff = np.var(diff, ddof=1)

    # Correction factor: accounts for n_test/n_train overlap
    correction = (1 / len(diff)) + (n_test / n_train)
    corrected_var = correction * var_diff

    t_stat = mean_diff / np.sqrt(corrected_var)
    p_value = 2 * stats.t.sf(np.abs(t_stat), df=len(diff) - 1)
    return t_stat, p_value, mean_diff

print("Corrected resampled t-test function defined.")
print("Positive mean_diff => champion has higher MSE (ensemble is better).")

### 3.6 Champion Decision (Regression)

The best-performing ensemble method (Simple Average or Weighted Average) is compared against the A4 champion. If the ensemble exceeds R² = 0.6356, the model is saved as `aimoscores_ensemble_A5.pkl`.

The best ensemble selection logic:
- Best aggregation method: `weighted` if weighted R² > average R², else `average`
- Saved only if it outperforms the A4 champion

---

## 4. Classification Ensemble (A5_Classification_Ensemble)

### 4.1 Problem Setup

- **Target**: `WeakestLink` — the movement category with the highest deviation score across 14 NASM categories.
- **Features**: Merged movement features from `aimoscores.csv` + weakest link labels from `scores_and_weaklink.csv`.
- **Class imbalance**: Addressed using `class_weight='balanced'` and `class_weight='balanced_subsample'`.
- **A4 Champion baseline**: Random Forest with weighted F1 = **0.6110**.

### 4.2 Data Preparation

In [ ]:
# Classification setup (from A5_Classification_Ensemble.ipynb)
RANDOM_STATE = 42
N_SPLITS     = 5
CHAMPION_F1  = 0.6110   # Sprint 4 benchmark

# 14 WeakestLink categories
weaklink_categories = [
    'ExcessiveForwardLean', 'ForwardHead', 'LeftArmFallForward',
    'LeftAsymmetricalWeightShift', 'LeftHeelRises', 'LeftKneeMovesInward',
    'LeftKneeMovesOutward', 'LeftShoulderElevation', 'RightArmFallForward',
    'RightAsymmetricalWeightShift', 'RightHeelRises', 'RightKneeMovesInward',
    'RightKneeMovesOutward', 'RightShoulderElevation',
]

# WeakestLink = the category with the highest deviation score
# weaklink_scores_df['WeakestLink'] = weaklink_scores_df[weaklink_categories].idxmax(axis=1)

print(f"Number of classes: {len(weaklink_categories)}")
print(f"Sprint 4 champion F1: {CHAMPION_F1}")

### 4.3 Ensemble Strategies

Four ensemble strategies were designed and evaluated using **5-fold stratified cross-validation**:

#### Ensemble 1 – Hard Voting
Each base classifier casts a vote; the class with the most votes wins. Base classifiers: Random Forest, Logistic Regression, XGBoost, LightGBM, KNN (k=7), LDA.

#### Ensemble 2 – Soft Voting  
Same base classifiers, but predictions are combined via averaged class probabilities. Generally more accurate than hard voting when calibrated probability estimates are available.

#### Ensemble 3 – Bootstrap Bagging on LDA  
`BaggingClassifier` wrapping `LinearDiscriminantAnalysis` (50 estimators, 80% sample size, 90% feature subset). Demonstrates how bagging can stabilise a weak linear model.

#### Ensemble 4 – Stacking (LR meta-learner)  
Base classifiers: Random Forest, Logistic Regression, KNN, LDA. Meta-learner: `LogisticRegression` trained on out-of-fold predictions (5-fold CV). The meta-learner learns *how to combine* base model outputs, replacing simple voting with a learned aggregation function.

In [ ]:
# Ensemble model definitions (from A5_Classification_Ensemble.ipynb)
from sklearn.ensemble import (
    RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
# import xgboost as xgb
# import lightgbm as lgb

# ---- Ensemble 4: Stacking ----
stacking = StackingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(n_estimators=100, max_depth=15,
                                       min_samples_split=5, min_samples_leaf=2,
                                       class_weight='balanced_subsample',
                                       random_state=RANDOM_STATE, n_jobs=-1)),
        ('lr',  LogisticRegression(max_iter=1000, class_weight='balanced',
                                   random_state=RANDOM_STATE)),
        ('knn', KNeighborsClassifier(n_neighbors=7)),
        ('lda', LinearDiscriminantAnalysis()),
    ],
    final_estimator=LogisticRegression(
        C=1.0, max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    cv=5,
    passthrough=False,
    n_jobs=-1,
)

print("Stacking classifier defined with LR meta-learner.")

### 4.4 Cross-Validation Results

All models were evaluated with 5-fold **StratifiedKFold** cross-validation on the training set, using weighted F1-score as the primary metric.

| Model | CV F1 (mean) | CV F1 (std) | CV Accuracy | CV Precision | CV Recall |
|-------|-------------|------------|------------|-------------|----------|
| A4 Champion – Random Forest | ~0.6110 | — | — | — | — |
| Hard Voting | runtime | runtime | runtime | runtime | runtime |
| Soft Voting | runtime | runtime | runtime | runtime | runtime |
| Bootstrap Bagging (LDA) | runtime | runtime | runtime | runtime | runtime |
| Stacking (LR meta) | runtime | runtime | runtime | runtime | runtime |

> *The bar chart below (produced at runtime) visually compares all approaches, with the red dashed line marking the Sprint 4 champion.*

### 4.5 Statistical Significance Tests

The corrected resampled t-test was applied for each ensemble vs the A4 champion (same implementation as the regression track).

| Ensemble | t-stat | p-value | Better than Champion? |
|----------|--------|---------|----------------------|
| Hard Voting | runtime | runtime | runtime |
| Soft Voting | runtime | runtime | runtime |
| Bootstrap Bagging (LDA) | runtime | runtime | runtime |
| Stacking (LR meta) | runtime | runtime | runtime |

### 4.6 Final Test Set Results

All models were retrained on the full training set and evaluated on the held-out 20% test split.

| Model | Test F1 | Test Accuracy | Test Precision | Test Recall |
|-------|---------|--------------|---------------|------------|
| Best Ensemble (champion) | runtime | runtime | runtime | runtime |
| A4 Champion – Random Forest | ~0.6110 | — | — | — |

### 4.7 Champion Decision (Classification)

The top-ranked ensemble by CV F1 is selected as the new champion and saved to `models/ensemble_classification_champion.pkl`. The artifact includes the model, scaler, feature columns, CV metrics, test metrics, and improvement percentage vs Sprint 4.

---
## 5. Sprint Summary

### 5.1 What Was Done

| Component | Owner Track | Description |
|-----------|------------|-------------|
| `CorrelationFilter.py` | Shared | Custom sklearn transformer removing highly correlated features |
| Regression Ensemble | Regression track | 4 base models (Lasso, Ridge, RF, GB) × 4 bootstrap samples, simple + weighted averaging |
| Classification Ensemble | Classification track | Hard Voting, Soft Voting, Bagging (LDA), Stacking (LR meta) |
| Statistical Testing | Both tracks | Corrected resampled t-test (Nadeau & Bengio 2003) |
| Champion Deployment | Both tracks | Pickle artifacts saved if ensemble outperforms A4 champion |

### 5.2 Key Design Decisions

**Regression:** Bootstrap-based diversity was the primary source of independence between base models. Weighted averaging was used as the aggregation method with weights derived from CV-R² scores, giving better-performing models proportionally more influence.

**Classification:** A broader range of diversity strategies was explored — algorithm diversity (RF, LR, XGB, LGB, KNN, LDA), voting schemes (hard vs soft), and a stacking approach where a meta-learner replaces manual aggregation. Class imbalance was consistently addressed with `class_weight='balanced'`.

**Statistical rigor:** The Nadeau-Bengio correction was applied in both tracks rather than a naive t-test, accounting for the overlap between cross-validation folds (correction factor ≈ 1.25 for this dataset).

### 5.3 Limitations and Next Steps

- Feature subset diversity (angle-only, NASM-only) was defined but ultimately the final configuration used full features for all base models in the regression track. Future iterations could test whether feature-diverse ensembles further reduce error.
- The classification stacking approach used `passthrough=False`, meaning the meta-learner only sees predicted class probabilities, not the original features. Including raw features (`passthrough=True`) could be explored.
- More ensemble members (e.g., 8–10 base models) could be evaluated to assess the accuracy-variance tradeoff more thoroughly.